In [1]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
sys.path.append(PROJECT_ROOT)

print(PROJECT_ROOT)

/Users/yongyili/EnvHealthLondon


In [2]:
from config.settings import *
print(GCP_PROJECT)
print(GCS_BUCKET)

env-health-london-2026
env-health-london-data-2026-katy


In [7]:
import requests
import pandas as pd
import time
from io import StringIO

BASE = "https://api.erg.ic.ac.uk/AirQuality"

def get_laqn_data(site_code, species, start, end, retries=3):
    url = (f"{BASE}/Data/SiteSpecies/SiteCode={site_code}"
           f"/SpeciesCode={species}"
           f"/StartDate={start}"
           f"/EndDate={end}"
           f"/csv")

    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=90)

            if r.status_code == 200 and len(r.content) > 200:
                return pd.read_csv(StringIO(r.text))
            else:
                print(f"No data/status {r.status_code}: {site_code} {species} {start}-{end}")
                return None

        except requests.exceptions.ReadTimeout:
            print(f"Timeout: {site_code} {start}-{end}, retry {attempt + 1}/{retries}")
            time.sleep(3)

        except requests.exceptions.RequestException as e:
            print(f"Request failed: {site_code} {start}-{end}: {e}")
            return None

    return None

In [12]:
frames = []
failed_requests = []

for i, row in sites.iterrows():
    site_code = row["@SiteCode"]
    
    for yr in range(2018, 2024):
        print(f"Downloading site {i+1}/{len(sites)}: {site_code}, year {yr}")
        
        df = get_laqn_data(
            site_code,
            "NO2",
            f"{yr}-01-01",
            f"{yr}-12-31"
        )
        
        if df is not None:
            try:
                # 🔹 FIX COLUMN NAME
                value_col = df.columns[1]
                df = df.rename(columns={value_col: "no2"})
                
                # 🔹 ADD METADATA
                df["site_code"] = site_code
                df["borough"] = row.get("@LocalAuthorityName", "Unknown")
                df["latitude"] = float(row.get("@Latitude", 0))
                df["longitude"] = float(row.get("@Longitude", 0))
                
                frames.append(df)
            
            except Exception as e:
                print(f"Column error for {site_code} {yr}: {e}")
                failed_requests.append((site_code, yr))
        
        else:
            failed_requests.append((site_code, yr))
        
        time.sleep(0.5)

# 🔹 SUMMARY BEFORE CONCAT
print("Successful dataframes:", len(frames))
print("Failed requests:", len(failed_requests))

# 🔹 SAFE CONCAT
if len(frames) == 0:
    raise ValueError("No LAQN data downloaded. Check API endpoint.")
else:
    laqn_raw = pd.concat(frames, ignore_index=True)
    
    # 🔹 SAVE DATA
    laqn_raw.to_parquet(f"{DATA_DIR}/laqn_raw.parquet")
    
    print(f"Downloaded {len(laqn_raw):,} hourly readings")
    print(f"Saved to: {DATA_DIR}/laqn_raw.parquet")

Successful dataframes: 1530
Failed requests: 0
Downloaded 13,373,730 hourly readings
Saved to: /Users/yongyili/EnvHealthLondon/data/laqn_raw.parquet


In [13]:
laqn_raw.head()
laqn_raw.info()
laqn_raw.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13373730 entries, 0 to 13373729
Data columns (total 6 columns):
 #   Column              Dtype  
---  ------              -----  
 0   MeasurementDateGMT  object 
 1   no2                 float64
 2   site_code           object 
 3   borough             object 
 4   latitude            float64
 5   longitude           float64
dtypes: float64(3), object(3)
memory usage: 612.2+ MB


,no2,latitude,longitude
count,4.523349e+06,1.337373e+07,1.337373e+07
mean,3.248704e+01,5.149465e+01,-1.219404e-01
std,2.374288e+01,5.995388e-02,1.464217e-01
min,-8.200000e+00,5.131456e+01,-4.608260e-01
25%,1.470000e+01,5.146390e+01,-2.158901e-01
50%,2.660000e+01,5.149506e+01,-1.270270e-01
75%,4.440000e+01,5.152257e+01,-2.259733e-02
max,9.635000e+02,5.166864e+01,2.253596e-01


In [ ]:
# Phase 2: Process the first dataset for Noise

In [14]:
import geopandas as gpd

noise = gpd.read_file(f"{DATA_DIR}/Road_Noise_Lden_England_Round_3.geojson")

noise.head()

,noiseclass,geometry
0,>=75.0,"POLYGON ((168245.000 24325.000, 168255.000 243..."
1,>=75.0,"POLYGON ((466965.000 458415.000, 466965.000 45..."
2,>=75.0,"POLYGON ((166875.000 25315.000, 166865.000 253..."
3,>=75.0,"POLYGON ((420145.000 253935.000, 420135.000 25..."
4,>=75.0,"POLYGON ((466965.000 458415.000, 466965.000 45..."


In [15]:
noise.shape
noise.columns

Index(['noiseclass', 'geometry'], dtype='object')

In [16]:
def parse_noise_class(value):
    value = str(value).replace("dB", "").strip()
    
    if ">=" in value:
        return float(value.replace(">=", "").strip())
    
    if ">" in value:
        return float(value.replace(">", "").strip())
    
    if "-" in value:
        low, high = value.split("-")
        return (float(low) + float(high)) / 2
    
    return float(value)

noise["lden_mean"] = noise["noiseclass"].apply(parse_noise_class)

noise[["noiseclass", "lden_mean"]].head()

,noiseclass,lden_mean
0,>=75.0,75.0
1,>=75.0,75.0
2,>=75.0,75.0
3,>=75.0,75.0
4,>=75.0,75.0


In [17]:
noise["lden_mean"].describe()

count    1.596360e+06
mean     6.372992e+01
std      5.983902e+00
min      5.745000e+01
25%      5.745000e+01
50%      6.245000e+01
75%      6.745000e+01
max      7.500000e+01
Name: lden_mean, dtype: float64

In [21]:
import geopandas as gpd

boroughs = gpd.read_file(f"{DATA_DIR}/London_Borough_Excluding_MHW.shp")

print(boroughs.shape)
print(boroughs.columns)
print(boroughs.crs)

boroughs.head()

(33, 8)
Index(['NAME', 'GSS_CODE', 'HECTARES', 'NONLD_AREA', 'ONS_INNER', 'SUB_2009',
       'SUB_2006', 'geometry'],
      dtype='object')
PROJCS["OSGB36 / British National Grid",GEOGCS["OSGB36",DATUM["Ordnance_Survey_of_Great_Britain_1936",SPHEROID["Airy 1830",6377563.396,299.3249646,AUTHORITY["EPSG","7001"]],AUTHORITY["EPSG","6277"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",49],PARAMETER["central_meridian",-2],PARAMETER["scale_factor",0.999601272],PARAMETER["false_easting",400000],PARAMETER["false_northing",-100000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]


,NAME,GSS_CODE,HECTARES,NONLD_AREA,ONS_INNER,SUB_2009,SUB_2006,geometry
0,Kingston upon Thames,E09000021,3726.117,0.000,F,None,None,"POLYGON ((516401.600 160201.800, 516407.300 16..."
1,Croydon,E09000008,8649.441,0.000,F,None,None,"POLYGON ((535009.200 159504.700, 535005.500 15..."
2,Bromley,E09000006,15013.487,0.000,F,None,None,"POLYGON ((540373.600 157530.400, 540361.200 15..."
3,Hounslow,E09000018,5658.541,60.755,F,None,None,"POLYGON ((521975.800 178100.000, 521967.700 17..."
4,Ealing,E09000009,5554.428,0.000,F,None,None,"POLYGON ((510253.500 182881.600, 510249.900 18..."


In [22]:
boroughs = boroughs[["GSS_CODE", "NAME", "geometry"]].rename(
    columns={
        "GSS_CODE": "borough_code",
        "NAME": "borough_name"
    }
)

boroughs = boroughs.to_crs("EPSG:27700")

In [23]:
boroughs.columns

Index(['borough_code', 'borough_name', 'geometry'], dtype='object')

In [24]:
boroughs.head()

,borough_code,borough_name,geometry
0,E09000021,Kingston upon Thames,"POLYGON ((516401.600 160201.800, 516407.300 16..."
1,E09000008,Croydon,"POLYGON ((535009.200 159504.700, 535005.500 15..."
2,E09000006,Bromley,"POLYGON ((540373.600 157530.400, 540361.200 15..."
3,E09000018,Hounslow,"POLYGON ((521975.800 178100.000, 521967.700 17..."
4,E09000009,Ealing,"POLYGON ((510253.500 182881.600, 510249.900 18..."


In [25]:
noise = noise.to_crs(boroughs.crs)

In [26]:
noise_london = gpd.sjoin(noise, boroughs, how="inner", predicate="intersects")

In [28]:
noise_borough = (
    noise_london
    .groupby("borough_name")["lden_mean"]
    .mean()
    .reset_index()
)

noise_borough.head()

,borough_name,lden_mean
0,Barking and Dagenham,62.939922
1,Barnet,63.562162
2,Bexley,63.082413
3,Brent,64.076461
4,Bromley,63.344279


In [29]:
noise_london.columns

Index(['noiseclass', 'geometry', 'lden_mean', 'index_right', 'borough_code',
       'borough_name'],
      dtype='object')

In [30]:
noise_borough.to_parquet(f"{DATA_DIR}/noise_borough.parquet")

In [ ]:
# Phase 2: process the second dataset greenspace

In [31]:
greenspace = gpd.read_file(f"{DATA_DIR}/opgrsp_gb.gpkg")

print(greenspace.shape)
print(greenspace.columns)
print(greenspace.crs)

greenspace.head()

(355705, 4)
Index(['id', 'access_type', 'ref_to_greenspace_site', 'geometry'], dtype='object')
EPSG:27700


,id,access_type,ref_to_greenspace_site,geometry
0,2212623F-BD96-4866-8B1C-4E583537454E,Motor Vehicle And Pedestrian,49E9C778-7418-A491-E063-8CCAA00A0EF3,POINT (523775.670 173549.560)
1,C8A4CBFA-C4C5-4D55-9901-8B9F1912053A,Pedestrian,49E9C867-8F15-A491-E063-8CCAA00A0EF3,POINT (386970.010 401941.450)
2,AA0ED80B-D03E-4D67-AC90-0ED06B5DD5B5,Pedestrian,49E9C86F-0B11-A491-E063-8CCAA00A0EF3,POINT (392009.300 172795.690)
3,5D9B93D6-6E65-4522-9686-D2ED170F6ED9,Motor Vehicle And Pedestrian,49E9C76C-1A5A-A491-E063-8CCAA00A0EF3,POINT (287737.320 60766.690)
4,B7236565-B0E4-489A-85F3-B24764D263EA,Pedestrian,49E9C86E-DD57-A491-E063-8CCAA00A0EF3,POINT (493454.550 289211.940)


In [32]:
greenspace.columns

Index(['id', 'access_type', 'ref_to_greenspace_site', 'geometry'], dtype='object')

In [34]:
greenspace["access_type"].value_counts(dropna=False)

access_type
Pedestrian                      283810
Motor Vehicle And Pedestrian     71323
Motor Vehicle                      572
Name: count, dtype: int64

In [35]:
greenspace["access_type"].unique()

array(['Motor Vehicle And Pedestrian', 'Pedestrian', 'Motor Vehicle'],
      dtype=object)

In [36]:
ACCESSIBLE_TYPES = [
    "Pedestrian",
    "Motor Vehicle And Pedestrian"
]

greenspace_public = greenspace[
    greenspace["access_type"].isin(ACCESSIBLE_TYPES)
].copy()

print(greenspace_public.shape)
greenspace_public["access_type"].value_counts()

(355133, 4)


access_type
Pedestrian                      283810
Motor Vehicle And Pedestrian     71323
Name: count, dtype: int64

In [37]:
greenspace_public = greenspace_public.to_crs("EPSG:27700")
boroughs = boroughs.to_crs("EPSG:27700")

greenspace_london = gpd.clip(greenspace_public, boroughs)

print(greenspace_london.shape)
greenspace_london.head()

(23854, 4)


,id,access_type,ref_to_greenspace_site,geometry
322937,70D40586-FAEE-474E-814D-E1BFC5A7387F,Motor Vehicle And Pedestrian,49E9C779-D043-A491-E063-8CCAA00A0EF3,POINT (524472.780 160593.590)
298566,8CB70C7F-6E9C-4B0D-91A7-07D099254710,Pedestrian,49E9C874-6E59-A491-E063-8CCAA00A0EF3,POINT (531736.270 156850.530)
340114,30B8C1BE-D9C3-4FB6-AEFE-92602018756F,Pedestrian,49E9C874-6E59-A491-E063-8CCAA00A0EF3,POINT (531853.230 156859.600)
164333,2BB108EE-B4E5-455C-9BC3-C2D67BE57241,Pedestrian,49E9C874-6E59-A491-E063-8CCAA00A0EF3,POINT (531940.300 156928.700)
53856,5B237618-98FA-4EBF-B72D-CF91595015C9,Pedestrian,49E9C874-6E59-A491-E063-8CCAA00A0EF3,POINT (531950.730 156935.200)


In [38]:
greenspace_london["green_area_m2"] = greenspace_london.geometry.area

/opt/anaconda3/envs/envhealth/lib/python3.10/site-packages/geopandas/geodataframe.py:1528: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [39]:
greenspace_joined = gpd.sjoin(
    greenspace_london[["green_area_m2", "geometry"]],
    boroughs[["borough_name", "geometry"]],
    how="inner",
    predicate="intersects"
)

greenspace_joined.head()

,green_area_m2,geometry,index_right,borough_name
322937,0.0,POINT (524472.780 160593.590),18,Sutton
298566,0.0,POINT (531736.270 156850.530),1,Croydon
340114,0.0,POINT (531853.230 156859.600),1,Croydon
164333,0.0,POINT (531940.300 156928.700),1,Croydon
53856,0.0,POINT (531950.730 156935.200),1,Croydon


In [40]:
import fiona

fiona.listlayers(f"{DATA_DIR}/opgrsp_gb.gpkg")

['access_point', 'greenspace_site']

In [41]:
greenspace_sites = gpd.read_file(
    f"{DATA_DIR}/opgrsp_gb.gpkg",
    layer="greenspace_site"
)

print(greenspace_sites.shape)
print(greenspace_sites.columns)
print(greenspace_sites.geom_type.value_counts())
greenspace_sites.head()

(165978, 7)
Index(['id', 'function', 'distinctive_name_1', 'distinctive_name_2',
       'distinctive_name_3', 'distinctive_name_4', 'geometry'],
      dtype='object')
MultiPolygon    165978
Name: count, dtype: int64


,id,function,distinctive_name_1,distinctive_name_2,distinctive_name_3,distinctive_name_4,geometry
0,49E9C6F0-83ED-A491-E063-8CCAA00A0EF3,Play Space,None,None,None,None,"MULTIPOLYGON (((297417.920 145120.050, 297429...."
1,49E9C71E-3CBA-A491-E063-8CCAA00A0EF3,Religious Grounds,St Mary Magdalen's Church,None,None,None,"MULTIPOLYGON (((576082.060 118226.790, 576052...."
2,49E9C795-0EE4-A491-E063-8CCAA00A0EF3,Religious Grounds,Contin Parish Church,None,None,None,"MULTIPOLYGON (((245692.920 855781.250, 245698...."
3,49E9C763-2EBD-A491-E063-8CCAA00A0EF3,Tennis Court,None,None,None,None,"MULTIPOLYGON (((442188.770 430186.170, 442193...."
4,49E9C6F0-F421-A491-E063-8CCAA00A0EF3,Playing Field,None,None,None,None,"MULTIPOLYGON (((420054.330 408151.430, 420173...."


In [42]:
fiona.listlayers(f"{DATA_DIR}/opgrsp_gb.gpkg")

['access_point', 'greenspace_site']

In [43]:
greenspace_sites = greenspace_sites.to_crs("EPSG:27700")
boroughs = boroughs.to_crs("EPSG:27700")

In [44]:
greenspace_sites["function"].value_counts().head(30)

function
Play Space                                45034
Religious Grounds                         22460
Public Park Or Garden                     22106
Playing Field                             21855
Other Sports Facility                     16303
Allotments Or Community Growing Spaces    13778
Cemetery                                   7748
Tennis Court                               7173
Bowling Green                              6541
Golf Course                                2980
Name: count, dtype: int64

In [45]:
PUBLIC_FUNCTIONS = [
    "Public Park Or Garden",
    "Playing Field",
    "Play Space",
    "Allotments Or Community Growing Spaces",
    "Bowling Green",
    "Religious Grounds",
    "Cemetery",
    "Golf Course",
    "Tennis Court"
]

greenspace_public = greenspace_sites[
    greenspace_sites["function"].isin(PUBLIC_FUNCTIONS)
].copy()

print(greenspace_public.shape)
greenspace_public["function"].value_counts()

(149675, 7)


function
Play Space                                45034
Religious Grounds                         22460
Public Park Or Garden                     22106
Playing Field                             21855
Allotments Or Community Growing Spaces    13778
Cemetery                                   7748
Tennis Court                               7173
Bowling Green                              6541
Golf Course                                2980
Name: count, dtype: int64

In [46]:
greenspace_london = gpd.clip(greenspace_public, boroughs).copy()

print(greenspace_london.shape)
greenspace_london.head()

(9522, 7)


,id,function,distinctive_name_1,distinctive_name_2,distinctive_name_3,distinctive_name_4,geometry
33413,49E9C70A-D0A6-A491-E063-8CCAA00A0EF3,Religious Grounds,Westerham Hill Baptist Church,None,None,None,"POLYGON ((543312.470 157201.540, 543280.060 15..."
6372,49E9C70A-CB2C-A491-E063-8CCAA00A0EF3,Golf Course,Cherry Lodge Golf Club,None,None,None,"POLYGON ((543211.260 159124.000, 543226.660 15..."
146100,49E9C70A-D259-A491-E063-8CCAA00A0EF3,Play Space,None,None,None,None,"POLYGON ((544612.570 159735.230, 544669.300 15..."
107427,49E9C70A-D24C-A491-E063-8CCAA00A0EF3,Tennis Court,None,None,None,None,"POLYGON ((544567.030 159830.430, 544585.410 15..."
40566,49E9C746-DEE0-A491-E063-8CCAA00A0EF3,Public Park Or Garden,Downe Camp,None,None,None,"POLYGON ((543016.900 159934.600, 543028.880 15..."


In [47]:
greenspace_london["green_area_m2"] = greenspace_london.geometry.area

In [48]:
greenspace_joined = gpd.sjoin(
    greenspace_london[["green_area_m2", "geometry"]],
    boroughs[["borough_name", "geometry"]],
    how="inner",
    predicate="intersects"
)

greenspace_joined.head()

,green_area_m2,geometry,index_right,borough_name
33413,3346.57275,"POLYGON ((543312.470 157201.540, 543280.060 15...",2,Bromley
6372,441166.02125,"POLYGON ((543211.260 159124.000, 543226.660 15...",2,Bromley
146100,1069.66100,"POLYGON ((544612.570 159735.230, 544669.300 15...",2,Bromley
107427,664.59055,"POLYGON ((544567.030 159830.430, 544585.410 15...",2,Bromley
40566,48097.51370,"POLYGON ((543016.900 159934.600, 543028.880 15...",2,Bromley


In [49]:
greenspace_borough = (
    greenspace_joined
    .groupby("borough_name")["green_area_m2"]
    .sum()
    .reset_index()
)

greenspace_borough.head()

,borough_name,green_area_m2
0,Barking and Dagenham,6.590169e+06
1,Barnet,1.748205e+07
2,Bexley,8.659319e+06
3,Brent,6.242091e+06
4,Bromley,1.858615e+07


In [50]:
greenspace_borough.sort_values("green_area_m2", ascending=False).head(10)

,borough_name,green_area_m2
26,Richmond upon Thames,2.858851e+07
31,Wandsworth,2.126137e+07
15,Havering,1.905986e+07
4,Bromley,1.858615e+07
7,Croydon,1.831190e+07
1,Barnet,1.748205e+07
20,Kingston upon Thames,1.703662e+07
25,Redbridge,1.700287e+07
23,Merton,1.354176e+07
9,Enfield,1.248222e+07


In [51]:
greenspace_borough.to_parquet(f"{DATA_DIR}/greenspace_borough.parquet")

In [ ]:
# Accessible greenspace was defined as OS Open Greenspace polygons with pedestrian access, including “Pedestrian” and “Motor Vehicle And Pedestrian” categories. Motor-vehicle-only access was excluded because it does not represent ordinary walkable public access.

In [ ]:
# Phase 2: Process the third dataset for IMD

In [66]:
imd = pd.read_csv(f"{DATA_DIR}/imd_full.csv")

imd_score_col = "Index of Multiple Deprivation (IMD) Score"
imd_rank_col = "Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)"
imd_decile_col = "Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)"
borough_col = "Local Authority District name (2019)"

# Make sure numeric columns are actually numeric
imd[imd_score_col] = pd.to_numeric(imd[imd_score_col], errors="coerce")
imd[imd_rank_col] = pd.to_numeric(imd[imd_rank_col], errors="coerce")
imd[imd_decile_col] = pd.to_numeric(imd[imd_decile_col], errors="coerce")

# Check decile values
print(imd[imd_decile_col].value_counts(dropna=False).sort_index())

Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)
1     3284
2     3284
3     3285
4     3284
5     3285
6     3284
7     3284
8     3285
9     3284
10    3285
Name: count, dtype: int64


In [67]:
imd_borough = (
    imd
    .groupby(borough_col)
    .agg(
        imd_score_mean=(imd_score_col, "mean"),
        imd_rank_mean=(imd_rank_col, "mean"),
        imd_decile_mean=(imd_decile_col, "mean"),
        pct_most_deprived=(imd_decile_col, lambda x: (x == 1).mean() * 100)
    )
    .reset_index()
    .rename(columns={borough_col: "borough_name"})
)

london_borough_names = boroughs["borough_name"].unique()

imd_borough_london = imd_borough[
    imd_borough["borough_name"].isin(london_borough_names)
].copy()

print(imd_borough_london.shape)
imd_borough_london.sort_values("pct_most_deprived", ascending=False).head(10)

(33, 5)


,borough_name,imd_score_mean,imd_rank_mean,imd_decile_mean,pct_most_deprived
112,Hackney,32.868201,7420.611111,2.743056,11.111111
117,Haringey,27.587200,11291.917241,3.903448,9.655172
139,Kensington and Chelsea,22.019301,15419.339806,5.213592,8.737864
30,Brent,25.199855,12034.705202,4.161850,5.780347
94,Enfield,25.403448,12698.519126,4.382514,5.464481
138,Islington,27.706041,10296.512195,3.642276,4.878049
8,Barking and Dagenham,32.883473,7244.500000,2.681818,3.636364
248,Southwark,26.041289,11546.481928,4.024096,3.012048
151,Lewisham,26.869361,10781.378698,3.763314,2.958580
175,Newham,29.771079,8650.018293,3.091463,2.439024


In [68]:
imd_borough.shape

(317, 5)

In [70]:
imd_borough_london.to_parquet(f"{DATA_DIR}/imd_borough.parquet")

In [ ]:
# Phase 2: Process LAQN data

In [59]:
import pandas as pd

laqn_raw = pd.read_parquet(f"{DATA_DIR}/laqn_raw.parquet")

print(laqn_raw.shape)
print(laqn_raw.columns)
laqn_raw.head()

(13373730, 6)
Index(['MeasurementDateGMT', 'no2', 'site_code', 'borough', 'latitude',
       'longitude'],
      dtype='object')


,MeasurementDateGMT,no2,site_code,borough,latitude,longitude
0,2018-01-01 00:00,NaN,TD0,Richmond,51.424304,-0.345715
1,2018-01-01 01:00,NaN,TD0,Richmond,51.424304,-0.345715
2,2018-01-01 02:00,NaN,TD0,Richmond,51.424304,-0.345715
3,2018-01-01 03:00,NaN,TD0,Richmond,51.424304,-0.345715
4,2018-01-01 04:00,NaN,TD0,Richmond,51.424304,-0.345715


In [60]:
laqn = laqn_raw.copy()

# Convert datetime
laqn["MeasurementDateGMT"] = pd.to_datetime(
    laqn["MeasurementDateGMT"],
    errors="coerce"
)

# Convert NO2 to numeric
laqn["no2"] = pd.to_numeric(laqn["no2"], errors="coerce")

# Remove invalid rows
laqn = laqn.dropna(subset=["MeasurementDateGMT", "no2", "borough"])

# Remove physically invalid / extreme values
laqn = laqn[(laqn["no2"] >= 0) & (laqn["no2"] < 300)]

# Add year and month
laqn["year"] = laqn["MeasurementDateGMT"].dt.year
laqn["month"] = laqn["MeasurementDateGMT"].dt.month

print(laqn.shape)
laqn[["MeasurementDateGMT", "no2", "borough", "year", "month"]].head()

(4521403, 8)


,MeasurementDateGMT,no2,borough,year,month
104894,2018-01-01 02:00:00,10.8,Barking and Dagenham,2018,1
104895,2018-01-01 03:00:00,10.3,Barking and Dagenham,2018,1
104896,2018-01-01 04:00:00,8.2,Barking and Dagenham,2018,1
104897,2018-01-01 05:00:00,8.2,Barking and Dagenham,2018,1
104898,2018-01-01 06:00:00,10.3,Barking and Dagenham,2018,1


In [61]:
covid_mask = (
    (laqn["year"] == 2020) &
    (laqn["month"].isin([3, 4, 5, 6]))
)

laqn_clean = laqn[~covid_mask].copy()

print("Before COVID exclusion:", laqn.shape)
print("After COVID exclusion:", laqn_clean.shape)

Before COVID exclusion: (4521403, 8)
After COVID exclusion: (4260245, 8)


In [62]:
laqn_borough = (
    laqn_clean
    .groupby(["borough", "year"])
    .agg(
        no2_mean=("no2", "mean"),
        no2_count=("no2", "count"),
        site_count=("site_code", "nunique")
    )
    .reset_index()
)

laqn_borough.head()

,borough,year,no2_mean,no2_count,site_count
0,Barking and Dagenham,2018,25.017930,11640,2
1,Barking and Dagenham,2019,21.215058,11263,2
2,Barking and Dagenham,2020,19.560107,11566,2
3,Barking and Dagenham,2021,18.603705,16141,2
4,Barking and Dagenham,2022,18.815202,16768,2


In [63]:
laqn_clean.to_parquet(f"{DATA_DIR}/laqn_clean.parquet")
laqn_borough.to_parquet(f"{DATA_DIR}/laqn_borough.parquet")

print("Saved:")
print(f"{DATA_DIR}/laqn_clean.parquet")
print(f"{DATA_DIR}/laqn_borough.parquet")

Saved:
/Users/yongyili/EnvHealthLondon/data/laqn_clean.parquet
/Users/yongyili/EnvHealthLondon/data/laqn_borough.parquet


In [64]:
import os

os.listdir(DATA_DIR)

['laqn_borough.parquet',
 'London_Borough_Excluding_MHW.shp',
 'London_Borough_Excluding_MHW.shx',
 'laqn_raw.parquet',
 '.DS_Store',
 'greenspace_borough.parquet',
 'noise_borough.parquet',
 'opgrsp_gb.gpkg',
 'London_Borough_Excluding_MHW.dbf',
 'London_Borough_Excluding_MHW.prj',
 'imd_borough.parquet',
 'Road_Noise_Lden_England_Round_3.geojson',
 'laqn_clean.parquet',
 'imd_full.csv']

In [65]:
print("LAQN boroughs:", laqn_borough["borough"].nunique())
print("Years:", sorted(laqn_borough["year"].unique()))

laqn_borough.groupby("year")["borough"].nunique()

LAQN boroughs: 29
Years: [2018, 2019, 2020, 2021, 2022, 2023]


year
2018    28
2019    29
2020    29
2021    28
2022    29
2023    29
Name: borough, dtype: int64